This Jupyter Notebook is part of a **collection of scripts curated by Tarlan Ahadli** for the **2025 Spring Semester**, based on **projects provided by Prof. Hajder Levente**. These scripts cover a range of computational topics, including **computer vision, image processing, numerical computing, and matrix transformations**, serving as educational resources for academic use.

### **Usage Restrictions**
This material is **strictly for academic and educational purposes**.  
**Commercial use is prohibited.**

For any **issues, suggestions, or inquiries**, please contact:  
📧 **tarlanahad@gmail.com**


# **Table of Contents**

1. [Introduction to Perspective Camera](#section1)
2. [World, Camera, and Image Coordinate Systems](#section2)
3. [Extrinsic Parameters and World to Camera Transformation](#section3)
4. [Projection from Camera Coordinates to Image Plane](#section4)
5. [Python Implementation](#section5)



# **1. Introduction to Perspective Camera** <a id='section1'></a>

The perspective camera model is the foundation of 3D computer vision. It describes how a 3D point in the real world is mapped to a 2D image captured by a camera. This mapping is governed by:

- **Extrinsic parameters**: Define the transformation from the world coordinate system to the camera coordinate system.
- **Intrinsic parameters**: Define the transformation from the camera coordinate system to the image coordinate system.

The goal of this lesson is to understand the mathematical foundations of perspective projection and implement it in Python.



# **2. World, Camera, and Image Coordinate Systems** <a id='section2'></a>

A camera captures a 3D scene and projects it onto a 2D image plane. This process involves three coordinate systems:

### **2.1 World Coordinate System**
A **3D point in the real world** is represented as:
$$
X = \begin{bmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{bmatrix}
$$
This system is **global** and independent of the camera.

### **2.2 Camera Coordinate System**
The camera has its own **local coordinate system**, with its origin at the camera center.
A point in the camera frame is:
$$
X_C = \begin{bmatrix} X_C \\ Y_C \\ Z_C \\ 1 \end{bmatrix}
$$
The relationship between world and camera coordinates is defined by the **extrinsic parameters**.

### **2.3 Image Coordinate System**
After projection onto the image plane, the 2D image coordinates are:
$$
(u, v)
$$
This transformation is defined by the **intrinsic parameters**.



# **3. Extrinsic Parameters and World to Camera Transformation** <a id='section3'></a>

The extrinsic parameters define the transformation from the **world coordinate system** to the **camera coordinate system**.

### **3.1 Rotation and Translation**
A 3D point in world coordinates is transformed into the camera frame using:
$$
X_C = R (X_w - t)
$$
where:
- **\( R \) is a 3×3 rotation matrix** that aligns the world coordinate system with the camera.
- **\( t \) is a 3×1 translation vector** that shifts the origin of the world system to the camera's position.

### **3.2 Homogeneous Representation**
Using homogeneous coordinates, we write:
$$
X_C = R \begin{bmatrix} I | -t \end{bmatrix} X_w
$$
This allows us to handle rotation and translation in a single matrix multiplication.



# **4. Projection from Camera Coordinates to Image Plane** <a id='section4'></a>

Once a 3D point is in **camera coordinates**, we project it onto the image plane using the **intrinsic matrix**:

### **4.1 Intrinsic Matrix**
The intrinsic parameters define how 3D camera coordinates are mapped to 2D image coordinates:
$$
K = \begin{bmatrix} f k_u & 0 & u_0 \\ 0 & f k_v & v_0 \\ 0 & 0 & 1 \end{bmatrix}
$$
where:
- **\( f \)** is the focal length.
- **\( k_u, k_v \)** scale the image pixels.
- **\( (u_0, v_0) \)** is the image center.

### **4.2 Perspective Projection**
A 3D point in camera coordinates is projected as:
$$
\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} \sim K X_C
$$
Expanding:
$$
\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \begin{bmatrix} \frac{f k_u}{Z_C} X_C + u_0 \\ \frac{f k_v}{Z_C} Y_C + v_0 \\ 1 \end{bmatrix}
$$



# **5. Python Implementation** <a id='section5'></a>

The following Python code implements this projection model in Google Colab using OpenCV and widgets for visualization.


In [3]:
!wget https://cv.inf.elte.hu/wp-content/uploads/2025/03/PointClouds.zip
!unzip -o PointClouds.zip

--2026-10-13 09:30:20--  https://cv.inf.elte.hu/wp-content/uploads/2025/03/PointClouds.zip
Resolving cv.inf.elte.hu (cv.inf.elte.hu)... 157.181.166.208
Connecting to cv.inf.elte.hu (cv.inf.elte.hu)|157.181.166.208|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4144644 (4.0M) [application/zip]
Saving to: ‘PointClouds.zip.1’

PointClouds.zip.1   100%[===================>]   3.95M  10.8MB/s    in 0.4s    

2026-10-13 09:30:20 (10.8 MB/s) - ‘PointClouds.zip.1’ saved [4144644/4144644]

Archive:  PointClouds.zip
  inflating: __MACOSX/._PointClouds  
  inflating: PointClouds/plane.xyz   
  inflating: __MACOSX/PointClouds/._plane.xyz  
  inflating: PointClouds/street.xyz  
  inflating: __MACOSX/PointClouds/._street.xyz  
  inflating: PointClouds/BeforeCross.xyz  
  inflating: __MACOSX/PointClouds/._BeforeCross.xyz  
  inflating: PointClouds/DoubleLines.xyz  
  inflating: __MACOSX/PointClouds/._DoubleLines.xyz  
  inflating: PointClouds/Tunnel.xyz  
  inflating: __MAC

In [4]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import ipywidgets as widgets
from IPython.display import display, clear_output

# Camera class to handle intrinsic and extrinsic parameters
class Camera:
    def __init__(self, fku, fkv, u0, v0):
        """
        Initializes the Camera with intrinsic parameters.
        - fku, fkv: focal length in pixels (scaling factors)
        - u0, v0: principal point (center of the image)
        """
        self.K = np.eye(3, dtype=np.float64)  # Intrinsic matrix
        self.K[0, 0] = fku  # Scaling factor in x-direction
        self.K[1, 1] = fkv  # Scaling factor in y-direction
        self.K[0, 2] = u0  # Principal point in x-direction
        self.K[1, 2] = v0  # Principal point in y-direction
        self.R = np.eye(3, dtype=np.float64)  # Rotation matrix
        self.T = np.zeros((3, 1), dtype=np.float64)  # Translation vector

    def set_extrinsics(self, phi, theta, r):
        """
        Sets the extrinsic parameters based on spherical coordinates.
        - phi: azimuthal angle (horizontal rotation)
        - theta: polar angle (vertical rotation)
        - r: distance from the origin
        """
        # Convert spherical coordinates to Cartesian coordinates
        tx = np.cos(phi) * np.sin(theta)
        ty = np.sin(phi) * np.sin(theta)
        tz = np.cos(theta)

        # Set the translation vector
        self.T = np.array([[tx * r], [ty * r], [tz * r]], dtype=np.float64)

        # Define the camera's orientation
        Z = np.array([-tx, -ty, -tz])  # Optical axis
        Z /= np.linalg.norm(Z)  # Normalize to unit vector

        Y = np.array([ty, -tx, 0])  # Y-axis of the camera
        Y /= np.linalg.norm(Y)  # Normalize

        # Ensure correct orientation of the Y-axis
        if int(np.floor(theta / np.pi)) % 2 == 0:
            Y *= -1

        X = np.cross(Y, Z)  # X-axis of the camera (cross product)

        # Construct the rotation matrix
        self.R = np.vstack([X, Y, Z])

# Function to read a point cloud from a .xyz file
def read_point_cloud(file_name):
    """ Reads a 3D point cloud from a .xyz file """
    return np.genfromtxt(file_name, delimiter=" ", dtype=np.float64).reshape(-1, 3, 1)

# Function to project and draw 3D points on a 2D image
def draw_points(img, pc, K, R, T):
    """ Projects 3D points onto a 2D image and draws them """
    for p in pc:
        p_C = R @ (p - T)  # Transform point from world to camera coordinates

        # Assign color: white if in front of the camera, blue if behind
        color = (255, 255, 255) if p_C[2, 0] >= 0 else (255, 0, 0)

        # Project point onto image plane using intrinsic matrix
        p_C = K @ p_C
        p_C /= p_C[2, 0]  # Homogeneous division

        # Convert to pixel coordinates
        u, v = int(p_C[0, 0]), int(p_C[1, 0])

        # Draw point if within image bounds
        if 0 <= u < img.shape[1] and 0 <= v < img.shape[0]:
            cv2.circle(img, (u, v), 2, color, -1)

# Load point cloud from file
pc_file = "PointClouds/ChainBridge.xyz"
pc = read_point_cloud(pc_file)

# Create a blank image
img = np.zeros((600, 800, 3), dtype=np.uint8)

# Initialize the camera with focal length and image center
cam = Camera(1000, 1000, 400, 300)

# Set initial camera pose (orientation and distance)
phi, theta, r = 0.5, 1.0, 100.0
cam.set_extrinsics(phi, theta, r)

# Create an output widget for displaying images
output = widgets.Output()
display(output)

def update_display():
    """ Updates and redraws the projected 3D points on the image """
    img.fill(0)  # Clear the image
    draw_points(img, pc, cam.K, cam.R, cam.T)  # Draw updated points
    with output:
        clear_output(wait=True)  # Clear previous image output
        cv2_imshow(img)  # Display updated image

# Initial display of the projected point cloud
update_display()





def on_left_click(b):
    global phi
    phi -= 0.1
    cam.set_extrinsics(phi, theta, r)
    update_display()

def on_right_click(b):
    global phi
    phi += 0.1
    cam.set_extrinsics(phi, theta, r)
    update_display()

def on_up_click(b):
    global theta
    theta += 0.1
    cam.set_extrinsics(phi, theta, r)
    update_display()

def on_down_click(b):
    global theta
    theta -= 0.1
    cam.set_extrinsics(phi, theta, r)
    update_display()

def on_zoom_in_click(b):
    global r
    r /= 1.1
    cam.set_extrinsics(phi, theta, r)
    update_display()

def on_zoom_out_click(b):
    global r
    r *= 1.1
    cam.set_extrinsics(phi, theta, r)
    update_display()

btn_left = widgets.Button(description="Left")
btn_right = widgets.Button(description="Right")
btn_up = widgets.Button(description="Up")
btn_down = widgets.Button(description="Down")
btn_zoom_in = widgets.Button(description="Zoom In")
btn_zoom_out = widgets.Button(description="Zoom Out")

btn_left.on_click(on_left_click)
btn_right.on_click(on_right_click)
btn_up.on_click(on_up_click)
btn_down.on_click(on_down_click)
btn_zoom_in.on_click(on_zoom_in_click)
btn_zoom_out.on_click(on_zoom_out_click)

display(widgets.HBox([btn_left, btn_right, btn_up, btn_down, btn_zoom_in, btn_zoom_out]))



ModuleNotFoundError: No module named 'google'